# Baseline: AE-WGAN

Autoencoder with a Wasserstein GAN in its latent space (Generative deep learning for Internet of Things network traffic generation, PRDC 2020). Every flow is a sequence of 20 packets (truncated, or padded with a marker value); the generator produces latent codes that the decoder turns into sequences, and a sequence ends at the first padding value. The original work trains the autoencoder for 300 epochs; 100 epochs are used here.

Run by `scripts/run_matrix.py` after the proposed model of the same dataset and seed, whose `split.json` and `test.csv` it reuses. The packets are cleaned with the same function as for the proposed model, and the run stops if its test flows differ from `test.csv`.

In [ ]:
SEED         = 0
DATASET      = 'facebook_video'
RAW_PATH_STR = 'data/facebook_video.csv'
OUT_DIR      = 'runs/facebook_video/seed0'
MODEL_KEY    = 'aewgan'
TEST_SIZE    = 0.3              # not used: the split is read from split.json

In [ ]:
import os
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()   # run from the repository root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
from data_utils import atomic_write_csv, load_packets, load_split_checked, postprocess_synthetic

ART = Path(OUT_DIR) / 'artifacts' / MODEL_KEY
ART.mkdir(parents=True, exist_ok=True)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'
torch.use_deterministic_algorithms(True)
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEQ_LEN      = 20       # packets per sequence
LATENT_DIM   = 16       # latent dimension of the autoencoder
BATCH_SIZE   = 32
AE_EPOCHS    = 100      # autoencoder epochs
WGAN_EPOCHS  = 2000     # generator steps
CRITIC_ITERS = 5        # critic steps per generator step
LAMBDA_GP    = 10       # weight of the gradient penalty
LR_AE        = 1e-3     # Adam
LR_WGAN      = 5e-5     # RMSprop

In [ ]:
df = load_packets(RAW_PATH_STR)
df['payload_log']   = np.log1p(df['payload_length'])
df['time_diff_log'] = np.log1p(df['time_diff'])
train_flows, test_flows = load_split_checked(df, OUT_DIR)
train_mask = df['flow_id'].isin(train_flows)
scaler_pkt = StandardScaler().fit(df.loc[train_mask, ['payload_log', 'time_diff_log']])
df[['plog_z', 'dt_z']] = scaler_pkt.transform(df[['payload_log', 'time_diff_log']])


def create_joint_sequences(dataframe, feature_cols, seq_len, flows_list, pad_val=-10.0):
    # (N_flows, seq_len, 2) array; flows are truncated or padded with pad_val, which lies far
    # below the normalized data so that the network can recognize the padding
    sub_df = dataframe[dataframe['flow_id'].isin(flows_list)]
    grouped = sub_df.groupby('flow_id')[feature_cols].apply(lambda x: x.values.tolist()).to_dict()
    seqs, pad_token = [], [pad_val] * len(feature_cols)
    for fid in flows_list:
        if fid not in grouped:
            continue
        s = grouped[fid]
        if len(s) > seq_len:
            s = s[:seq_len]
        elif len(s) < seq_len:
            s = s + [pad_token] * (seq_len - len(s))
        seqs.append(s)
    return np.array(seqs, dtype=np.float32)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim=2, seq_len=20, latent_dim=16):
        super().__init__()
        self.seq_len = seq_len
        self.td_dense = nn.Linear(input_dim, 128)       # applied to every time step
        self.lstm1 = nn.LSTM(128, 128, batch_first=True)
        self.lstm2 = nn.LSTM(128, 64, batch_first=True)
        self.dense1 = nn.Linear(64, 32)
        self.bn1 = nn.BatchNorm1d(32)
        self.dense2 = nn.Linear(32, latent_dim)
        self.bn2 = nn.BatchNorm1d(latent_dim)
        self.tanh = nn.Tanh()

    def forward(self, x):
        x = self.tanh(self.td_dense(x))
        x, _ = self.lstm1(x)
        _, (h_n, _) = self.lstm2(x)
        x = self.tanh(self.bn1(self.dense1(h_n[-1])))
        return self.tanh(self.bn2(self.dense2(x)))


class Decoder(nn.Module):
    def __init__(self, latent_dim=16, seq_len=20, output_dim=2):
        super().__init__()
        self.seq_len = seq_len
        self.dense1 = nn.Linear(latent_dim, 32)
        self.bn1 = nn.BatchNorm1d(32)
        self.lstm1 = nn.LSTM(32, 64, batch_first=True)
        self.lstm2 = nn.LSTM(64, 128, batch_first=True)
        self.td_dense = nn.Linear(128, output_dim)
        self.tanh = nn.Tanh()

    def forward(self, z):
        x = self.tanh(self.bn1(self.dense1(z)))
        x = x.unsqueeze(1).repeat(1, self.seq_len, 1)
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        return self.td_dense(x)


class Autoencoder(nn.Module):
    def __init__(self, seq_len=20, latent_dim=16, feature_dim=2):
        super().__init__()
        self.encoder = Encoder(feature_dim, seq_len, latent_dim)
        self.decoder = Decoder(latent_dim, seq_len, feature_dim)

    def forward(self, x):
        return self.decoder(self.encoder(x))


class WGANGenerator(nn.Module):
    # noise (16) -> 4 x [dense 16, batch norm, tanh] -> dense 16 without activation
    def __init__(self, latent_dim):
        super().__init__()

        def block(in_feat, out_feat, normalize=True):
            layers = [nn.Linear(in_feat, out_feat)]
            if normalize:
                layers.append(nn.BatchNorm1d(out_feat))
            layers.append(nn.Tanh())
            return layers

        self.model = nn.Sequential(*block(latent_dim, 16), *block(16, 16), *block(16, 16), *block(16, 16),
                                   nn.Linear(16, latent_dim))

    def forward(self, z):
        return self.model(z)


class WGANCritic(nn.Module):
    # latent code (16) -> 4 x [dense 16, LeakyReLU 0.01] -> score
    def __init__(self, latent_dim):
        super().__init__()

        def block(in_feat, out_feat):
            return [nn.Linear(in_feat, out_feat), nn.LeakyReLU(0.01, inplace=True)]

        self.model = nn.Sequential(*block(latent_dim, 16), *block(16, 16), *block(16, 16), *block(16, 16),
                                   nn.Linear(16, 1))

    def forward(self, x):
        return self.model(x)


def compute_gradient_penalty(D, real_samples, fake_samples):
    alpha = torch.rand((real_samples.size(0), 1)).to(device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    d_interpolates = D(interpolates)
    fake = torch.ones((real_samples.shape[0], 1), requires_grad=False).to(device)
    gradients = autograd.grad(outputs=d_interpolates, inputs=interpolates, grad_outputs=fake,
                              create_graph=True, retain_graph=True, only_inputs=True)[0]
    gradients = gradients.view(gradients.size(0), -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

In [ ]:
def run_joint_generation_pipeline(train_flows, test_flows, df, scaler_pkt):
    X_train = create_joint_sequences(df, ['plog_z', 'dt_z'], SEQ_LEN, train_flows, pad_val=-10.0)
    dataloader = DataLoader(TensorDataset(torch.Tensor(X_train)), batch_size=BATCH_SIZE, shuffle=True)

    # 1. autoencoder; the padding is excluded from the reconstruction loss
    ae = Autoencoder(SEQ_LEN, LATENT_DIM, feature_dim=2).to(device)
    optimizer_ae = optim.Adam(ae.parameters(), lr=LR_AE)
    ae.train()
    for epoch in range(AE_EPOCHS):
        for batch in dataloader:
            real_seq = batch[0].to(device)
            optimizer_ae.zero_grad()
            recon = ae(real_seq)
            mask = (real_seq > -9.0).float()
            loss = (nn.MSELoss(reduction='none')(recon, real_seq) * mask).sum() / mask.sum()
            loss.backward()
            optimizer_ae.step()

    # 2. latent codes of the training sequences
    ae.eval()
    latent_vectors = []
    with torch.no_grad():
        for batch in dataloader:
            latent_vectors.append(ae.encoder(batch[0].to(device)).cpu().numpy())
    latent_loader = DataLoader(TensorDataset(torch.Tensor(np.vstack(latent_vectors))), batch_size=BATCH_SIZE, shuffle=True)

    # 3. WGAN with gradient penalty in the latent space
    generator = WGANGenerator(LATENT_DIM).to(device)
    critic = WGANCritic(LATENT_DIM).to(device)
    optimizer_G = optim.RMSprop(generator.parameters(), lr=LR_WGAN)
    optimizer_C = optim.RMSprop(critic.parameters(), lr=LR_WGAN)

    def infinite_loader(loader):
        while True:
            for batch in loader:
                yield batch

    data_iter = infinite_loader(latent_loader)
    for step in tqdm(range(WGAN_EPOCHS), desc='WGAN'):
        for _ in range(CRITIC_ITERS):
            real_z = next(data_iter)[0].to(device)
            optimizer_C.zero_grad()
            fake_z = generator(torch.randn(real_z.size(0), LATENT_DIM).to(device))
            real_validity = critic(real_z)
            fake_validity = critic(fake_z)
            gp = compute_gradient_penalty(critic, real_z, fake_z)
            d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + LAMBDA_GP * gp
            d_loss.backward()
            optimizer_C.step()
        optimizer_G.zero_grad()
        g_loss = -torch.mean(critic(generator(torch.randn(BATCH_SIZE, LATENT_DIM).to(device))))
        g_loss.backward()
        optimizer_G.step()

    # 4. one sequence per test flow
    generator.eval()
    ae.eval()
    num_to_generate, generated, curr = len(test_flows), [], 0
    with torch.no_grad():
        while curr < num_to_generate:
            rem = min(BATCH_SIZE, num_to_generate - curr)
            generated.append(ae.decoder(generator(torch.randn(rem, LATENT_DIM).to(device))).cpu().numpy())
            curr += rem
    torch.save(ae.state_dict(), ART / 'ae_weights_wgan.pth')
    torch.save(generator.state_dict(), ART / 'generator_weights_wgan.pth')
    joblib.dump(scaler_pkt, ART / 'scaler_pkt_wgan.pkl')
    return np.vstack(generated)


synth_joint_z = run_joint_generation_pipeline(train_flows, test_flows, df, scaler_pkt)

records = []
for i, fid in enumerate(tqdm(test_flows, desc='post-processing')):
    seq_joint = synth_joint_z[i]                            # (SEQ_LEN, 2), normalized
    inverse_data = scaler_pkt.inverse_transform(seq_joint)
    for t in range(SEQ_LEN):
        if seq_joint[t, 0] < -5.0:                          # padding value: end of the flow
            break
        payload = np.clip(np.expm1(inverse_data[t, 0]), 0.0, 1500.0)
        dt = max(np.expm1(inverse_data[t, 1]), 0.0)
        records.append([fid, round(payload), float(dt)])
synth_df = pd.DataFrame(records, columns=['flow_id', 'payload_length', 'time_diff'])

synth_df = postprocess_synthetic(synth_df)
atomic_write_csv(synth_df, Path(OUT_DIR) / f'{MODEL_KEY}.csv')
print(f'saved {len(synth_df):,} packets in {synth_df.flow_id.nunique()} flows')